In [0]:
from pyspark.sql.types import *
from pyspark.sql.functions import current_timestamp, col

BASE_PATH = "/Volumes/supply_chain_dev/lp_analytics/raw_sources"

# Bronze table names
BRONZE_SAP_ORDERS    = "supply_chain_dev.lp_analytics.bronze_sap_orders"
BRONZE_SAP_INVOICES  = "supply_chain_dev.lp_analytics.bronze_sap_invoices"
BRONZE_SF_ACCOUNTS   = "supply_chain_dev.lp_analytics.bronze_sf_accounts"
BRONZE_SF_CONTRACTS  = "supply_chain_dev.lp_analytics.bronze_sf_contracts"
BRONZE_TMS_SHIPMENTS = "supply_chain_dev.lp_analytics.bronze_tms_shipments"

print("Bronze table targets:")
print(f"  {BRONZE_SAP_ORDERS}")
print(f"  {BRONZE_SAP_INVOICES}")
print(f"  {BRONZE_SF_ACCOUNTS}")
print(f"  {BRONZE_SF_CONTRACTS}")
print(f"  {BRONZE_TMS_SHIPMENTS}")

Bronze table targets:
  supply_chain_dev.lp_analytics.bronze_sap_orders
  supply_chain_dev.lp_analytics.bronze_sap_invoices
  supply_chain_dev.lp_analytics.bronze_sf_accounts
  supply_chain_dev.lp_analytics.bronze_sf_contracts
  supply_chain_dev.lp_analytics.bronze_tms_shipments


In [0]:
from pyspark.sql.types import *
from pyspark.sql.functions import current_timestamp, col

BASE_PATH = "/Volumes/supply_chain_dev/lp_analytics/raw_sources"
BRONZE_SAP_ORDERS = "supply_chain_dev.lp_analytics.bronze_sap_orders"

sap_orders_schema = StructType([
    StructField("sap_order_id", StringType(), True),
    StructField("customer_id", StringType(), True),
    StructField("customer_name", StringType(), True),
    StructField("order_date", StringType(), True),
    StructField("service_type", StringType(), True),
    StructField("trade_lane", StringType(), True),
    StructField("incoterms", StringType(), True),
    StructField("freight_cost_usd", DoubleType(), True),
    StructField("revenue_usd", DoubleType(), True),
    StructField("margin_usd", DoubleType(), True),
    StructField("margin_pct", DoubleType(), True),
    StructField("weight_kg", DoubleType(), True),
    StructField("volume_cbm", DoubleType(), True),
    StructField("sap_division", StringType(), True),
    StructField("payment_terms", StringType(), True),
    StructField("order_status", StringType(), True),
    StructField("created_by", StringType(), True),
    StructField("source_system", StringType(), True),
    StructField("extract_date", StringType(), True)
])

sap_orders_source = BASE_PATH + "/sap/orders/"
sap_orders_checkpoint = BASE_PATH + "/checkpoints/sap/orders"
sap_orders_schema_loc = BASE_PATH + "/checkpoints/sap/orders_schema"

(spark.readStream
    .format("cloudFiles")
    .option("cloudFiles.format", "csv")
    .option("cloudFiles.schemaLocation", sap_orders_schema_loc)
    .option("header", "true")
    .schema(sap_orders_schema)
    .load(sap_orders_source)
    .withColumn("ingested_at", current_timestamp())
    .withColumn("source_file", col("_metadata.file_path"))
    .writeStream
    .format("delta")
    .option("checkpointLocation", sap_orders_checkpoint)
    .outputMode("append")
    .trigger(availableNow=True)
    .toTable(BRONZE_SAP_ORDERS)
)

print(f"SAP Orders Bronze: {spark.table(BRONZE_SAP_ORDERS).count()} records")

SAP Orders Bronze: 0 records


In [0]:
print(f"SAP Orders Bronze: {spark.table(BRONZE_SAP_ORDERS).count()} records")

SAP Orders Bronze: 300 records


In [0]:
BRONZE_SAP_INVOICES = "supply_chain_dev.lp_analytics.bronze_sap_invoices"

sap_invoices_schema = StructType([
    StructField("invoice_id", StringType(), True),
    StructField("sap_order_id", StringType(), True),
    StructField("customer_id", StringType(), True),
    StructField("invoice_date", StringType(), True),
    StructField("invoice_amount_usd", DoubleType(), True),
    StructField("payment_status", StringType(), True),
    StructField("days_to_pay", DoubleType(), True),
    StructField("source_system", StringType(), True),
    StructField("extract_date", StringType(), True)
])

sap_invoices_source = BASE_PATH + "/sap/invoices/"
sap_invoices_checkpoint = BASE_PATH + "/checkpoints/sap/invoices"
sap_invoices_schema_loc = BASE_PATH + "/checkpoints/sap/invoices_schema"

(spark.readStream
    .format("cloudFiles")
    .option("cloudFiles.format", "csv")
    .option("cloudFiles.schemaLocation", sap_invoices_schema_loc)
    .option("header", "true")
    .schema(sap_invoices_schema)
    .load(sap_invoices_source)
    .withColumn("ingested_at", current_timestamp())
    .withColumn("source_file", col("_metadata.file_path"))
    .writeStream
    .format("delta")
    .option("checkpointLocation", sap_invoices_checkpoint)
    .outputMode("append")
    .trigger(availableNow=True)
    .toTable(BRONZE_SAP_INVOICES)
)

print(f"SAP Invoices Bronze: {spark.table(BRONZE_SAP_INVOICES).count()} records")

SAP Invoices Bronze: 0 records


In [0]:
print(f"SAP Invoices Bronze: {spark.table(BRONZE_SAP_INVOICES).count()} records")

SAP Invoices Bronze: 250 records


In [0]:
BRONZE_SF_ACCOUNTS = "supply_chain_dev.lp_analytics.bronze_sf_accounts"

sf_accounts_schema = StructType([
    StructField("sf_account_id", StringType(), True),
    StructField("customer_id", StringType(), True),
    StructField("account_name", StringType(), True),
    StructField("segment", StringType(), True),
    StructField("region", StringType(), True),
    StructField("account_owner", StringType(), True),
    StructField("industry", StringType(), True),
    StructField("annual_revenue_usd", DoubleType(), True),
    StructField("employee_count", IntegerType(), True),
    StructField("account_status", StringType(), True),
    StructField("nps_score", IntegerType(), True),
    StructField("created_date", StringType(), True),
    StructField("last_activity_date", StringType(), True),
    StructField("source_system", StringType(), True),
    StructField("extract_date", StringType(), True)
])

sf_accounts_source = BASE_PATH + "/salesforce/accounts/"
sf_accounts_checkpoint = BASE_PATH + "/checkpoints/salesforce/accounts"
sf_accounts_schema_loc = BASE_PATH + "/checkpoints/salesforce/accounts_schema"

(spark.readStream
    .format("cloudFiles")
    .option("cloudFiles.format", "json")
    .option("cloudFiles.schemaLocation", sf_accounts_schema_loc)
    .schema(sf_accounts_schema)
    .load(sf_accounts_source)
    .withColumn("ingested_at", current_timestamp())
    .withColumn("source_file", col("_metadata.file_path"))
    .writeStream
    .format("delta")
    .option("checkpointLocation", sf_accounts_checkpoint)
    .outputMode("append")
    .trigger(availableNow=True)
    .toTable(BRONZE_SF_ACCOUNTS)
)

print(f"Salesforce Accounts Bronze: {spark.table(BRONZE_SF_ACCOUNTS).count()} records")

Salesforce Accounts Bronze: 0 records


In [0]:
print(f"Salesforce Accounts Bronze: {spark.table(BRONZE_SF_ACCOUNTS).count()} records")

Salesforce Accounts Bronze: 15 records


In [0]:
BRONZE_SF_CONTRACTS = "supply_chain_dev.lp_analytics.bronze_sf_contracts"

sf_contracts_schema = StructType([
    StructField("contract_id", StringType(), True),
    StructField("customer_id", StringType(), True),
    StructField("sf_account_id", StringType(), True),
    StructField("contract_type", StringType(), True),
    StructField("service_scope", StringType(), True),
    StructField("contract_value_usd", DoubleType(), True),
    StructField("start_date", StringType(), True),
    StructField("end_date", StringType(), True),
    StructField("contract_status", StringType(), True),
    StructField("renewal_probability_pct", IntegerType(), True),
    StructField("assigned_rep", StringType(), True),
    StructField("source_system", StringType(), True),
    StructField("extract_date", StringType(), True)
])

sf_contracts_source = BASE_PATH + "/salesforce/contracts/"
sf_contracts_checkpoint = BASE_PATH + "/checkpoints/salesforce/contracts"
sf_contracts_schema_loc = BASE_PATH + "/checkpoints/salesforce/contracts_schema"

(spark.readStream
    .format("cloudFiles")
    .option("cloudFiles.format", "json")
    .option("cloudFiles.schemaLocation", sf_contracts_schema_loc)
    .schema(sf_contracts_schema)
    .load(sf_contracts_source)
    .withColumn("ingested_at", current_timestamp())
    .withColumn("source_file", col("_metadata.file_path"))
    .writeStream
    .format("delta")
    .option("checkpointLocation", sf_contracts_checkpoint)
    .outputMode("append")
    .trigger(availableNow=True)
    .toTable(BRONZE_SF_CONTRACTS)
)

print(f"Salesforce Contracts Bronze: {spark.table(BRONZE_SF_CONTRACTS).count()} records")

Salesforce Contracts Bronze: 0 records


In [0]:
print(f"Salesforce Contracts Bronze: {spark.table(BRONZE_SF_CONTRACTS).count()} records")

Salesforce Contracts Bronze: 29 records


In [0]:
BRONZE_TMS_SHIPMENTS = "supply_chain_dev.lp_analytics.bronze_tms_shipments"

tms_schema = StructType([
    StructField("shipment_id", StringType(), True),
    StructField("sap_order_ref", StringType(), True),
    StructField("customer_id", StringType(), True),
    StructField("carrier", StringType(), True),
    StructField("service_type", StringType(), True),
    StructField("trade_lane", StringType(), True),
    StructField("origin_port", StringType(), True),
    StructField("destination_port", StringType(), True),
    StructField("ship_date", StringType(), True),
    StructField("eta", StringType(), True),
    StructField("actual_arrival", StringType(), True),
    StructField("transit_days_actual", IntegerType(), True),
    StructField("delay_days", IntegerType(), True),
    StructField("on_time_delivery", BooleanType(), True),
    StructField("shipment_status", StringType(), True),
    StructField("container_count", IntegerType(), True),
    StructField("weight_kg", DoubleType(), True),
    StructField("freight_charges_usd", DoubleType(), True),
    StructField("customs_cleared", BooleanType(), True),
    StructField("source_system", StringType(), True),
    StructField("extract_date", StringType(), True)
])

tms_source = BASE_PATH + "/tms/shipments/"
tms_checkpoint = BASE_PATH + "/checkpoints/tms/shipments"
tms_schema_loc = BASE_PATH + "/checkpoints/tms/shipments_schema"

(spark.readStream
    .format("cloudFiles")
    .option("cloudFiles.format", "json")
    .option("cloudFiles.schemaLocation", tms_schema_loc)
    .schema(tms_schema)
    .load(tms_source)
    .withColumn("ingested_at", current_timestamp())
    .withColumn("source_file", col("_metadata.file_path"))
    .writeStream
    .format("delta")
    .option("checkpointLocation", tms_checkpoint)
    .outputMode("append")
    .trigger(availableNow=True)
    .toTable(BRONZE_TMS_SHIPMENTS)
)

print(f"TMS Shipments Bronze: {spark.table(BRONZE_TMS_SHIPMENTS).count()} records")

TMS Shipments Bronze: 0 records


In [0]:
print(f"TMS Shipments Bronze: {spark.table(BRONZE_TMS_SHIPMENTS).count()} records")

TMS Shipments Bronze: 400 records


In [0]:
display(spark.sql("SHOW TABLES IN supply_chain_dev.lp_analytics"))

database,tableName,isTemporary
lp_analytics,bronze_sap_invoices,false
lp_analytics,bronze_sap_orders,false
lp_analytics,bronze_sf_accounts,false
lp_analytics,bronze_sf_contracts,false
lp_analytics,bronze_tms_shipments,false
